# 🌾 Rice Vision AI - Colab Backend Server
Notebook này khởi động FastAPI AI Backend trên Google Colab GPU (Tesla T4) với ngrok tunnel.

**Tự động hóa hoàn toàn:** Token và Domain được đọc trực tiếp từ file `AI_SERVICES/.env` trên Google Drive của bạn.

In [1]:
# Cell 1: Mount Google Drive & Cài đặt thư viện
from google.colab import drive
drive.mount('/content/drive')

!pip install fastapi uvicorn pyngrok python-multipart
!pip install ultralytics sahi opencv-python-headless joblib scikit-learn

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 13.6 MB/s eta 0:00:00


In [2]:
# Cell 2: Thiết lập đường dẫn & Tự động đọc file quản lý Token (.env)
import os
import sys

# === ĐƯỜNG DẪN DỰ ÁN TRÊN GOOGLE DRIVE CỦA BẠN ===
DRIVE_PROJECT = '/content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES'
AI_SERVICES_DIR = os.path.join(DRIVE_PROJECT, 'AI_SERVICES')
# =================================================

if AI_SERVICES_DIR not in sys.path:
    sys.path.insert(0, AI_SERVICES_DIR)

# Đọc file .env quản lý TOKEN
ENV_FILE = os.path.join(AI_SERVICES_DIR, '.env')
CONFIG = {}

if os.path.exists(ENV_FILE):
    with open(ENV_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                k, v = line.split('=', 1)
                CONFIG[k.strip()] = v.strip()
    print(f'✅ Đã đọc thành công file cấu hình: {ENV_FILE}')
    has_token = bool(CONFIG.get('NGROK_AUTH_TOKEN'))
    domain = CONFIG.get('NGROK_DOMAIN') or '(Tự sinh link ngẫu nhiên)'
    print(f'   - NGROK_AUTH_TOKEN : {"Đã cấu hình (Bảo mật)" if has_token else "CHƯA CÓ ⚠️ (Cần điền vào .env)"}')
    print(f'   - NGROK_DOMAIN     : {domain}')
else:
    print(f'⚠️ Không tìm thấy {ENV_FILE}. Vui lòng đảm bảo đã copy thư mục AI_SERVICES lên Drive.')

# Kiểm tra mã nguồn và file trọng số mô hình
print('\n--- Kiểm tra Model Trọng số ---')
check_files = [
    ('YOLOv8-seg', os.path.join(DRIVE_PROJECT, 'RESULTS/all-new-data-v1.yolov8_yolov8s-seg_trained/weights/best.pt')),
    ('DenseNet121', os.path.join(DRIVE_PROJECT, 'RESULTS/CNN_DenseNet121_Trained/best_v3_step2.keras')),
    ('ExtraTrees', os.path.join(DRIVE_PROJECT, 'LINEAR_REGRESSION_MODEL/models/best_tree_ensemble_model.joblib')),
    ('Scaler', os.path.join(DRIVE_PROJECT, 'LINEAR_REGRESSION_MODEL/models/scaler.joblib')),
]
for name, p in check_files:
    st = 'OK' if os.path.exists(p) else 'MISSING'
    print(f'[{st}] {name}: {os.path.basename(p)}')

✅ Đã đọc thành công file cấu hình: /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/AI_SERVICES/.env
   - NGROK_AUTH_TOKEN : Đã cấu hình (Bảo mật)
   - NGROK_DOMAIN     : provolone-duress-probably.ngrok-free.dev

--- Kiểm tra Model Trọng số ---
[OK] YOLOv8-seg: best.pt
[OK] DenseNet121: best_v3_step2.keras
[OK] ExtraTrees: best_tree_ensemble_model.joblib
[OK] Scaler: scaler.joblib


In [3]:
# Cell 3: Khởi động FastAPI server + ngrok tunnel (Sử dụng Token từ file .env)
import nest_asyncio
nest_asyncio.apply()

from pyngrok import ngrok
import uvicorn
import os
import asyncio

token = CONFIG.get('NGROK_AUTH_TOKEN', '').strip()
domain = CONFIG.get('NGROK_DOMAIN', '').strip()

if not token:
    raise ValueError(
        '\n\n❌ CHƯA CÓ NGROK_AUTH_TOKEN!\n'
        'Vui lòng mở file AI_SERVICES/.env trên máy hoặc trên Drive,\n'
        'điền token của bạn vào dòng: NGROK_AUTH_TOKEN=your_token_here rồi chạy lại cell này.'
    )

# Nạp token từ file .env
ngrok.set_auth_token(token)
print('🔑 Đã nạp Ngrok Auth Token thành công từ file .env!')

os.chdir(AI_SERVICES_DIR)
from app import app

PORT = int(CONFIG.get('PORT_AI', 8000))

# Mở ngrok tunnel
if domain:
    clean_domain = domain.replace('https://', '').replace('http://', '').strip()
    print(f'🌐 Đang kết nối với Static Domain: {clean_domain}...')
    tunnel = ngrok.connect(PORT, domain=clean_domain)
else:
    print('🌐 Đang kết nối với Dynamic URL...')
    tunnel = ngrok.connect(PORT)

public_url = tunnel.public_url
print(f'\n{"="*60}')
print(f'  🚀 NGROK PUBLIC URL: {public_url}')
print(f'  Web Frontend sẽ tự động nhận diện URL này qua file .env')
print(f'{"="*60}\n')

# Tự động đồng bộ URL mới vào file .env trên Drive
try:
    if os.path.exists(ENV_FILE):
        with open(ENV_FILE, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        with open(ENV_FILE, 'w', encoding='utf-8') as f:
            updated = False
            for line in lines:
                if line.startswith('AI_SERVER_URL='):
                    f.write(f'AI_SERVER_URL={public_url}\n')
                    updated = True
                else:
                    f.write(line)
            if not updated:
                f.write(f'\nAI_SERVER_URL={public_url}\n')
        print('💾 Đã lưu AI_SERVER_URL vào file .env!')
except Exception as e:
    print(f'⚠️ Lưu .env thất bại: {e}')

# Khởi chạy máy chủ suy luận AI (Sử dụng cách chạy Server thủ công để tránh lỗi loop_factory với nest_asyncio)
config = uvicorn.Config(app, host='0.0.0.0', port=PORT, loop="asyncio")
server = uvicorn.Server(config)

loop = asyncio.get_event_loop()
loop.create_task(server.serve())

🔑 Đã nạp Ngrok Auth Token thành công từ file .env!


/content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/AI_SERVICES/app.py:51: UserWarning: 
[CANH BAO MOI TRUONG] Python 3.13 co the khong tuong thich hoan hao voi TensorFlow/PyTorch.
Khuyen nghi su dung Python 3.10, 3.11 hoac 3.12 (64-bit) de dam bao on dinh tuyet doi!

  warnings.warn(


🌐 Đang kết nối với Static Domain: provolone-duress-probably.ngrok-free.dev...

  🚀 NGROK PUBLIC URL: https://provolone-duress-probably.ngrok-free.dev
  Web Frontend sẽ tự động nhận diện URL này qua file .env

💾 Đã lưu AI_SERVER_URL vào file .env!


<Task pending name='Task-1' coro=<Server.serve() running at /usr/local/lib/python3.13/dist-packages/uvicorn/server.py:79>>